In [8]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

Layer Normalization

In [1]:
import torch
import torch.nn as nn

In [2]:
class LayerNorm(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(out_dim))
        self.shift = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim = True)
        var = x.var(dim=-1, keepdim = True, unbiased = False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        return self.scale * x_norm + self.shift

In [3]:
batch = torch.randn(2,5)
batch

layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())
x = layer(batch)

In [4]:
torch.set_printoptions(sci_mode=False)

In [5]:
ln = LayerNorm(6)
out = ln(x)
out.mean(dim = -1)

tensor([0.0000, 0.0000], grad_fn=<MeanBackward1>)

GeLU Activation Layer

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x + 0.044715 * torch.pow(x, 3))))

In [19]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        self.layers(x)

In [20]:
inp = torch.rand(2, 3, GPT_CONFIG_124M["emb_dim"])
inp

tensor([[[0.9059, 0.6783, 0.6732,  ..., 0.2214, 0.3409, 0.5525],
         [0.2965, 0.8803, 0.7554,  ..., 0.9851, 0.7833, 0.7576],
         [0.7780, 0.9137, 0.6723,  ..., 0.3415, 0.5563, 0.2683]],

        [[0.9081, 0.3807, 0.2512,  ..., 0.5363, 0.9314, 0.4001],
         [0.5550, 0.6534, 0.2944,  ..., 0.0349, 0.2103, 0.7426],
         [0.0175, 0.4246, 0.1434,  ..., 0.5905, 0.6462, 0.2091]]])

In [21]:
ffn = FeedForward(GPT_CONFIG_124M)

ffn(inp)

TypeError: sqrt(): argument 'input' (position 1) must be Tensor, not float